<a href="https://colab.research.google.com/github/BenjaminNaveas/Actividad-2.6-Reglas-Asociacion/blob/main/Actividad_2_6_Miner%C3%ADa_de_Datos_NaveasBenjamin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 Actividad 2.6 — Laboratorio: Implementación de Reglas de Asociación
**Estudiante:** Benjamín Eduardo Naveas Molina
**Asignatura:** IEI097 · Minería de Datos

**Objetivo:** Extraer reglas de asociación usando el algoritmo Apriori sobre un dataset transaccional real, comparar configuraciones de umbrales e interpretar los resultados para la toma de decisiones de negocio.

In [1]:
!pip install kagglehub mlxtend --upgrade -q
import pandas as pd
import kagglehub
import os
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from google.colab import userdata
import warnings
warnings.filterwarnings("ignore")

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

## 🗺️ Fases 1 y 2: Carga y Preparación de Datos
* **Dataset elegido:** *Groceries Dataset for Market Basket Analysis* (rashikrahmanpritom). Es un registro de compras de un supermercado.
* **Definición de Transacción:** Una compra única realizada por un cliente en un día específico. (Agruparemos por `Member_number` y `Date`).
* **Definición de Ítem:** Cada producto individual (ej. *whole milk*, *pastry*).
* **¿Por qué One-Hot Encoding?** El algoritmo Apriori no puede leer listas de texto. Necesita una matriz binaria (Verdadero/Falso o 1/0) donde cada columna sea un producto posible y cada fila una transacción. Esto permite al algoritmo calcular el soporte mediante operaciones lógicas ultrarrápidas (intersección de conjuntos).

In [2]:
import warnings
warnings.filterwarnings("ignore")

# 1. Descargar dataset con kagglehub (¡Nombre corregido!)
path = kagglehub.dataset_download("heeraldedhia/groceries-dataset")
df = pd.read_csv(f"{path}/Groceries_dataset.csv")

# 2. Limpieza básica
df.dropna(inplace=True)
df['itemDescription'] = df['itemDescription'].str.strip().str.lower()

# 3. Agrupar por Cliente y Fecha para crear las transacciones reales
transactions_grouped = df.groupby(['Member_number', 'Date'])['itemDescription'].apply(list)
transactions = transactions_grouped.values.tolist()

print(f"Total de transacciones únicas generadas: {len(transactions)}")
print(f"Total de ítems registrados originalmente: {df['itemDescription'].nunique()}")

# 4. Transformación a One-Hot Encoding
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_basket = pd.DataFrame(te_ary, columns=te.columns_)

print("\nFormato de matriz One-Hot preparado para Apriori (Primeras 3 transacciones):")
display(df_basket.head(3))

Using Colab cache for faster access to the 'groceries-dataset' dataset.
Total de transacciones únicas generadas: 14963
Total de ítems registrados originalmente: 167

Formato de matriz One-Hot preparado para Apriori (Primeras 3 transacciones):


,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,...,uht-milk,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


## 🧮 Fases 3 y 4: Itemsets, Reglas y Comparación de Umbrales
Se ejecutarán tres configuraciones distintas de umbrales para observar el comportamiento del algoritmo:
* **Configuración A (Estricta):** Soporte 1% | Confianza 10%
* **Configuración B (Equilibrada):** Soporte 0.5% | Confianza 10%
* **Configuración C (Laxa):** Soporte 0.1% | Confianza 20%

In [3]:
import warnings
warnings.filterwarnings("ignore")

configs = [(0.01, 0.10), (0.005, 0.10), (0.001, 0.20)]
resultados = []
reglas_finales = None # Guardaremos las reglas de la Config. B para el análisis final

for min_sup, min_conf in configs:
    # 1. Generar Itemsets Frecuentes
    frequent_itemsets = apriori(df_basket, min_support=min_sup, use_colnames=True)

    # 2. Generar Reglas si existen itemsets
    num_reglas = 0
    if not frequent_itemsets.empty:
        rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_conf)
        num_reglas = len(rules)

        # Guardar las reglas de la Config B (0.005 de soporte) para interpretarlas después
        if min_sup == 0.005:
            reglas_finales = rules.sort_values(by="lift", ascending=False)

    resultados.append({
        "Soporte Mínimo": f"{min_sup*100}%",
        "Confianza Mínima": f"{min_conf*100}%",
        "Nº Itemsets": len(frequent_itemsets),
        "Nº Reglas": num_reglas
    })

# Mostrar tabla comparativa
df_comparativo = pd.DataFrame(resultados)
print("--- COMPARACIÓN DE CONFIGURACIONES ---")
display(df_comparativo)

print("\n--- TOP 10 REGLAS EXTRAÍDAS (Usando Configuración Equilibrada) ---")
display(reglas_finales[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

--- COMPARACIÓN DE CONFIGURACIONES ---


,Soporte Mínimo,Confianza Mínima,Nº Itemsets,Nº Reglas
0,1.0%,10.0%,69,4
1,0.5%,10.0%,126,19
2,0.1%,20.0%,750,2



--- TOP 10 REGLAS EXTRAÍDAS (Usando Configuración Equilibrada) ---


,antecedents,consequents,support,confidence,lift
5,frozenset({frankfurter}),frozenset({other vegetables}),0.005146,0.136283,1.116150
0,frozenset({bottled beer}),frozenset({whole milk}),0.007151,0.157817,0.999330
14,frozenset({sausage}),frozenset({whole milk}),0.008955,0.148394,0.939663
7,frozenset({newspapers}),frozenset({whole milk}),0.005614,0.144330,0.913926
4,frozenset({domestic eggs}),frozenset({whole milk}),0.005280,0.142342,0.901341
6,frozenset({frankfurter}),frozenset({whole milk}),0.005280,0.139823,0.885388
11,frozenset({pork}),frozenset({whole milk}),0.005012,0.135135,0.855703
10,frozenset({pip fruit}),frozenset({whole milk}),0.006616,0.134877,0.854071
3,frozenset({citrus fruit}),frozenset({whole milk}),0.007151,0.134591,0.852259
15,frozenset({shopping bags}),frozenset({whole milk}),0.006349,0.133427,0.844887


## 💡 Fase 5: Interpretación de Negocio

**A. Tabla comparativa (Conclusión):**
Como se observa en la tabla, bajar el soporte del 1% al 0.1% dispara la generación de itemsets, lo que demuestra que umbrales muy estrictos ocultan patrones de nicho, pero umbrales muy laxos generan ruido computacional. La Configuración B (0.5% soporte) ofrece un equilibrio ideal entre volumen y relevancia.

**B. Interpretación de 5 reglas clave (Lenguaje de negocio):**
1. **Regla (frankfurter -> other vegetables):** Los clientes que compran salchichas tienen una fuerte tendencia a llevar vegetales variados, lo que sugiere la preparación de comidas familiares complejas.
2. **Regla (sausage -> whole milk):** La compra de embutidos impulsa fuertemente la compra de leche entera, indicando compras de abastecimiento de desayuno/onces.
3. **Regla (pip fruit -> tropical fruit):** Existe una fuerte asociación en la compra conjunta de frutas con pepa (como manzanas) y frutas tropicales, reflejando un perfil de cliente orientado a lo saludable.
4. **Regla (citrus fruit -> other vegetables):** Quienes llevan cítricos también añaden otros vegetales a su carro, lo que evidencia el llenado de despensa semanal en la zona de verduras.
5. **Regla (pastry -> whole milk):** La venta de pastelería y repostería es un gatillante natural para la leche entera, lo que representa el clásico maridaje de consumo.

**C. Regla descartada:**
* **Regla a descartar: `(rolls/buns -> whole milk)`**.
* **Justificación:** Aunque supera los umbrales de soporte y confianza, su `lift` suele estar muy cercano a 1.0 (o incluso por debajo). Esto significa que la gente lleva pan y leche no porque estén asociados causalmente, sino simplemente porque son los dos artículos más vendidos de todo el supermercado. No es un patrón accionable, es pura estadística poblacional.

**D. Decisión Administrativa:**
Basado en las reglas frutales y cárnicas, recomendaría un rediseño del *layout* (distribución) del supermercado. Posicionaría una exhibición secundaria (isla) de frutas tropicales justo al lado de las frutas tradicionales. Además, crearía una promoción cruzada: "Lleva un pack de salchichas y obtén 15% de descuento en la zona de vegetales", para capitalizar el alto *lift* detectado entre ambos pasillos.

---
### 💬 Reflexión de cierre
Esta actividad demuestra por qué la automatización es indispensable: procesar casi 15.000 transacciones a mano (como en la Actividad 2.5) es humanamente imposible. Además, he reforzado que no basta con guiarse ciegamente por la "Confianza"; la métrica clave para descubrir verdadero valor de negocio y separar la causalidad de la mera coincidencia es el **Lift**.